In [1]:
import sys
sys.path.insert(0, '..')
sys.path.insert(0, '../exposure_to_hazards/')

from config import DATA_DIR

In [50]:
from pathlib import Path
from climada.entity import Exposures
import numpy as np
import pandas as pd

exposures_dict = {}
path_population_files = DATA_DIR / "population"

path_exposures = path_population_files / "worldpop/climada_exposures"

exposures_dict = {}
for age in ["0_1", "all", "65_70_75_80"]:
    exposures_dict[age]= {}
    for year in np.arange(2003, 2023):
        year = str(year)
        exposures_dict[age][year]= Exposures.from_hdf5(path_exposures / f"{age}_era5_025_compatible_{year}.hdf5")
        exposures_dict[age][year].gdf["impf_cs"] = 1 
        exposures_dict[age][year].gdf['value'][exposures_dict[age][year].gdf['value']<0] = 0


In [56]:
geometry = exposures_dict[age][str(year)].gdf['geometry'] = exposures_dict[age][str(year)].gdf['geometry'].apply(
                lambda geom: Point(geom.x - 360 if geom.x > 180 else geom.x, geom.y)) 
for age in ["0_1", "all", "65_70_75_80"]:
    for year in np.arange(2003, 2023):
        exposures_dict[age][str(year)].gdf['geometry'] = geometry

In [54]:
exposures_dict[age]

{'2003': <climada.entity.exposures.base.Exposures at 0x30b148290>,
 '2004': <climada.entity.exposures.base.Exposures at 0x33fb51290>,
 '2005': <climada.entity.exposures.base.Exposures at 0x372c2fb50>,
 '2006': <climada.entity.exposures.base.Exposures at 0x46a184750>,
 '2007': <climada.entity.exposures.base.Exposures at 0x3110b34d0>,
 '2008': <climada.entity.exposures.base.Exposures at 0x30c9bd650>,
 '2009': <climada.entity.exposures.base.Exposures at 0x33d1e4d90>,
 '2010': <climada.entity.exposures.base.Exposures at 0x31d66d890>,
 '2011': <climada.entity.exposures.base.Exposures at 0x31d64c450>,
 '2012': <climada.entity.exposures.base.Exposures at 0x32de79c90>,
 '2013': <climada.entity.exposures.base.Exposures at 0x3e63660d0>,
 '2014': <climada.entity.exposures.base.Exposures at 0x31d64c490>,
 '2015': <climada.entity.exposures.base.Exposures at 0x32de6eb10>,
 '2016': <climada.entity.exposures.base.Exposures at 0x32de37390>,
 '2017': <climada.entity.exposures.base.Exposures at 0x34929c1

In [4]:
(7.8/(exposures_dict['all']['2020'].gdf.value.sum()/1e9))

0.9799007436716637

In [ ]:
#  1. merge exposure gdf with adm1 for any age any year (we just need the lat/lon points by admin1)

In [57]:
import geopandas as gpd
from shapely.geometry import Point


gdf_adm1 = gpd.read_file(path_population_files / "geoBoundariesCGAZ_ADM1/geoBoundariesCGAZ_ADM1.shp")
gdf_adm1 = gdf_adm1.to_crs("EPSG:4326")
gdf_adm1 =  gdf_adm1[gdf_adm1['shapeType']=='ADM1']

gdf_adm1 = gdf_adm1.rename(columns={'shapeGroup':'ISO3', 'shapeName':'ADM1_NAME', 'shapeID': 'ADM1_ID'})
gdf_adm1.to_file(path_population_files / "geoBoundariesCGAZ_ADM1_renamed/geoBoundariesCGAZ_ADM1.shp")

geometry = exposures_dict[age][str(year)].gdf.geometry

gdf = gpd.GeoDataFrame(exposures_dict[age][str(year)].gdf, crs="EPSG:4326", geometry=geometry)
gdf = gpd.sjoin(gdf, gdf_adm1, how="left", predicate='within')


In [ ]:
gdf

In [58]:
import pandas as pd
#geometry = [Point(xy) for xy in zip(exposures_dict[age][str(year)].gdf.longitude, exposures_dict[age][str(year)].gdf.latitude)]

for age in ['all', '0_1', '65_70_75_80']:
    gdf = gpd.GeoDataFrame(exposures_dict[age][str(year)].gdf, crs="EPSG:4326", geometry=geometry)
    merged = gpd.sjoin(gdf, gdf_adm1, how="left", predicate='within')
    #merged = merged.drop_duplicates(subset=['latitude', 'longitude'], keep='first')
    merged.to_csv(path_population_files / f"{age}_worldpop_admin_by_point.csv")

In [59]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# Assuming exposures_dict, year, and gdf_adm1 are already defined and loaded appropriately.

for age in ['all']:
    # Create GeoDataFrame from your dictionary data
    #geometry = [Point(xy) for xy in zip(exposures_dict[age][str(year)].gdf.longitude, exposures_dict[age][str(year)].gdf.latitude)]
    gdf = gpd.GeoDataFrame(exposures_dict[age][str(year)].gdf, crs="EPSG:4326", geometry=geometry)

    # Spatial join using 'within'
    within_join = gpd.sjoin(gdf, gdf_adm1, how="left", predicate='within')
    #within_join = within_join.drop_duplicates(subset=['latitude', 'longitude'], keep='first')
    #within_join.to_csv(path_population_files / f"{age}_worldpop_admin_by_point_within.csv")
    
    # Spatial join using 'intersects'
    intersects_join = gpd.sjoin(gdf, gdf_adm1, how="left", predicate='intersects')
    #intersects_join = intersects_join.drop_duplicates(subset=['latitude', 'longitude'], keep='first')
    #intersects_join.to_csv(path_population_files / f"{age}_worldpop_admin_by_point_intersects.csv")

    # Optionally, calculate and print total population from both joins to compare
    total_within = within_join['value'].sum()  # Replace 'population' with the correct column name for population data
    total_intersects = intersects_join['value'].sum()
    print(f"Total population for age {age} using 'within': {total_within}")
    print(f"Total population for age {age} using 'intersects' (duplicates dropped): {total_intersects}")
    intersects_join.to_csv(path_population_files / f"{age}_worldpop_admin_by_point_within.csv")


Total population for age all using 'within': 8227061449.619627
Total population for age all using 'intersects' (duplicates dropped): 8227402986.108569


In [60]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
import copy

# Assuming exposures_dict, gdf_adm1, and path_population_files are already defined and loaded appropriately.
# Assuming all years and age categories use the same geographic points.

# Create GeoDataFrame from your dictionary data (using any year and age since all are the same)
sample_year = 2020
sample_age = 'all'
#geometry = [Point(xy) for xy in zip(exposures_dict[sample_age][str(sample_year)].gdf.longitude, exposures_dict[sample_age][str(sample_year)].gdf.latitude)]
gdf = gpd.GeoDataFrame(exposures_dict[sample_age][str(sample_year)].gdf, crs="EPSG:4326", geometry=geometry)

# Spatial join using 'within'
joined_gdf = gpd.sjoin(gdf, gdf_adm1, how="left", predicate='within')

# Filter out points that have not been attributed to an admin area
attributed_points = joined_gdf[joined_gdf['ADM1_NAME'].notna()]
#geometry = [Point(x, y) for x, y in zip(annual_data.longitude, annual_data.latitude)]
# Proceed with analysis or data manipulation as needed
pop_by_adm1_by_year = {}
for age in ['all', '65_70_75_80', '0_1']:
    total_pop_by_year = []
    for year in range(2003, 2021):
        # Merge with annual data
        annual_data = copy.deepcopy(exposures_dict[age][str(year)].gdf)
        annual_data = annual_data.set_geometry(geometry)
        annual_data = pd.merge(annual_data, attributed_points[['ADM1_ID', 'ISO3', 'ADM1_NAME', 'geometry']], on='geometry', how='left')
        aggregated_data = annual_data[['value', 'ADM1_ID', 'ISO3', 'ADM1_NAME']].groupby(['ADM1_ID', 'ISO3', 'ADM1_NAME']).sum()
        aggregated_data['year'] = year
        total_pop_by_year.append(aggregated_data)

    pop_by_adm1_by_year[age] = pd.concat(total_pop_by_year).reset_index()
    #pop_by_adm1_by_year[age].to_csv(path_population_files / f"{age}_worldpop_by_admin1_by_year_2003_2021.csv")


In [ ]:
gdf_countries = gpd.read_file("/Users/szelie/OneDrive - ETH Zurich/data/lancet/admin_boundaries/Detailed_Boundary_ADM0")


In [ ]:
for age in ['all', '65_70_75_80', '0_1']:
    pop_by_iso3_by_year =  pop_by_adm1_by_year[age].groupby(['ISO3', 'year']).sum()
    # Save the results to CSV
    pop_by_iso3_by_year.reset_index()[['value', 'ISO3', 'year']].to_csv(path_population_files / f"{age}_worldpop_by_ISO3_by_year_2003_2021.csv")
    
# Final message
print("Population data by ISO3 has been processed and saved.")


In [ ]:
path_population_files / f"{age}_worldpop_by_ISO3_by_year_2000_2022.csv"